# SE1_data_access

**Thesis:** Near-Real-Time Streamflow and Nutrient Prediction for the Oulanka River Using SWAT+ and Open Meteorological APIs  
**Author:** Masters Student, Water, Energy and Environmental Engineering (WE3), University of Oulu  
**Repository:** [GitHub – oulanka-swatplus-prediction]

---

## Purpose
This notebook is **Step 1** of the four-part reproducible pipeline (SE1 → SE4).  
It downloads **all raw data** required to drive the SWAT+ model from stable, publicly accessible URLs following FAIR data principles (Findable, Accessible, Interoperable, Reusable).

The notebook is organised into three parts:

| Part | What | When to run |
|------|------|-------------|
| **A** | Static model inputs (TxtInOut, Shapes) from Zenodo | **Once**, or when you publish a new model version |
| **B** | Dynamic meteorological forcing | **Daily**, to keep the prediction current |
| **B-Option 1** | FMI point station (default) — real-time observations via WFS API + Harmonie NWP forecast | Daily |
| **B-Option 2** | FMI gridded 1 km analysis (alternative) — daily NetCDF files from a public S3 bucket | Daily |
| **B-Option 3** | ERA5 reanalysis (alternative) — monthly download via Copernicus CDS API | Monthly or as needed |

> **Run only one of B-Option 1, 2, or 3.** Set `FORCING_OPTION` below to choose. SE2 reads the output regardless of which option you used.

## Data sources

| # | Source | Protocol | Stable URL / Identifier | Variables | Auth required |
|---|--------|----------|------------------------|-----------|---------------|
| 1 | Zenodo | HTTPS REST API (Rule 3) | `https://zenodo.org/api/records/19207614` | `TxtInOut_test.zip`, `Shapes.zip` | Token (env var) |
| 2 | FMI Open Data | OGC WFS 2.0 API (Rule 4) | `http://opendata.fmi.fi/wfs` — fmisid=101887, 101932 | Daily pcp, Tmax, Tmin, hmd, slr | None |
| 3 | FMI Harmonie NWP | OGC WFS 2.0 API (Rule 4) | `http://opendata.fmi.fi/wfs` — forecast::harmonie | Temperature, Humidity, Precipitation1h, RadiationGlobal | None |
| 4 | FMI Gridded 1 km | Public S3 HTTP (Rule 3) | `http://fmi-gridded-obs-daily-1km.s3-website-eu-west-1.amazonaws.com/Netcdf/` | RRday, Tmax, Tmin, Rh, Globrad (NetCDF) | None |
| 5 | ERA5 Reanalysis | Copernicus CDS API (Rule 4) | `https://cds.climate.copernicus.eu` | 2m temperature, total precipitation, radiation, humidity | CDS API key (env var) |

## Authentication instructions (see also README)

**Zenodo (Part A):**  
Generate a Personal Access Token at https://zenodo.org/account/settings/applications/  
Store it as an environment variable — **never paste it into the notebook**:
```bash
export ZENODO_TOKEN="your_token_here"
```
If your record is open-access (public), no token is needed.

**Copernicus CDS (Option 3 only):**  
Register at https://cds.climate.copernicus.eu and follow https://cds.climate.copernicus.eu/how-to-api  
Store your UID and API key in `~/.cdsapirc` (the `cdsapi` library reads this file automatically — no hardcoding needed).

## Storage requirements
- `TxtInOut_test.zip`: 50–150 MB (check your Zenodo record).
- `Shapes.zip`: < 10 MB.
- FMI grid NetCDF (per variable per year): ~20–80 MB. 7 variables × 5 years ≈ 1–3 GB total.
- ERA5 (monthly, subsetted to Finland): typically 50–200 MB per request.
- FMI point / Harmonie: in-memory only, no large files written.

In [ ]:
!df -k ..
# Check available storage before downloading. FMI grid data can require 1-3 GB.

## Cell 1 — Load dependencies

In [ ]:
# Modify cell — add or remove imports as needed
from pathlib import Path
import os
import glob
import re
import time
import datetime
import zipfile
import shutil
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

# Option 1 only
from fmiopendata.wfs import download_stored_query   # pip install fmiopendata

# Option 2 only
import xarray as xr                                # pip install xarray netcdf4
from pyproj import Transformer                     # pip install pyproj

# Option 3 only
# import cdsapi                                    # pip install cdsapi  (uncomment if using ERA5)

print("Dependencies loaded successfully.")

## Cell 2 — Directory setup and user configuration

**Set `FORCING_OPTION` here to choose your meteorological data source.**  
All other settings are resolved automatically. Raw data lives outside the repository.

In [ ]:
# Automatically generated cell — resolve external directories
# Current working directory = project/notebooks
notebook_dir  = Path.cwd()
project_root  = notebook_dir.parent          # one level up from notebooks/
external_base = project_root.parent          # OUTSIDE the git repository

# Raw data directory outside repo (never committed to GitHub)
raw_data_dir = external_base / "oulanka_swatplus_rawdata"
raw_data_dir.mkdir(parents=True, exist_ok=True)

# ── USER SETTINGS ──────────────────────────────────────────────────────────────
# Choose your meteorological forcing option:
#   "FMI_POINT"  — Option 1: FMI station observations + Harmonie NWP (default)
#   "FMI_GRID"   — Option 2: FMI gridded 1 km analysis (alternative)
#   "ERA5"       — Option 3: ERA5 reanalysis (alternative)
FORCING_OPTION = "FMI_POINT"    # <-- change this to switch data source

# Zenodo record ID (update if you publish a new version)
ZENODO_RECORD_ID = "19207614"
ZENODO_FILES     = ["TxtInOut_test.zip", "Shapes.zip"]
ZENODO_DIR       = raw_data_dir / "zenodo_data"
ZENODO_DIR.mkdir(parents=True, exist_ok=True)

# Paths after Zenodo extraction (adjust if zip extracts with different folder names)
TXTINOUT_DIR = ZENODO_DIR / "TxtInOut_test"
SHAPE_DIR    = ZENODO_DIR / "Shapes"

# FMI point station IDs (Option 1)
FMI_STATION_ID_OBS = "101887"        # Kuusamo Airport — daily observations
FMI_STATION_ID_RAD = "101932"        # Kuusamo — radiation station
FORECAST_LATLON    = "66.364,29.316" # Oulanka outlet (lat, lon)

# FMI gridded data settings (Option 2)
FMI_GRID_S3_URL   = "http://fmi-gridded-obs-daily-1km.s3-website-eu-west-1.amazonaws.com/Netcdf/"
FMI_GRID_VARS     = ["RRday", "Tmax", "Tmin", "Rh", "Globrad"]   # variables to download
FMI_GRID_START_YR = 2022            # first year to download (update as needed)
FMI_GRID_DIR      = raw_data_dir / "FMI_Gridded_Data"
FMI_GRID_DIR.mkdir(parents=True, exist_ok=True)

# ERA5 settings (Option 3)
ERA5_AREA         = [72, 19, 58, 32]   # [North, West, South, East] — bounding box for Finland
ERA5_YEARS        = [str(y) for y in range(2015, datetime.datetime.now().year + 1)]
ERA5_DIR          = raw_data_dir / "ERA5"
ERA5_DIR.mkdir(parents=True, exist_ok=True)

# SWAT+ base directory (parent of working TxtInOut copies — set to Zenodo output)
BASE_DIR = str(ZENODO_DIR)

print(f"Forcing option selected : {FORCING_OPTION}")
print(f"Raw data directory      : {raw_data_dir.resolve()}")
print(f"Zenodo data directory   : {ZENODO_DIR.resolve()}")

---
# Part A — Static model inputs (Rule 2 + Rule 3)
## Cell 3 — Download TxtInOut and Shapes from Zenodo

**Run once.** The calibrated SWAT+ model configuration (`TxtInOut_test.zip`) and GIS shapefiles (`Shapes.zip`) are stored on Zenodo with a permanent DOI-based identifier, satisfying Rules 2 and 3 for data accessibility.

The token is read from the `ZENODO_TOKEN` environment variable — it is never written into this file.  
For open-access (public) records, set `ZENODO_TOKEN` is not required and can be omitted.

In [ ]:
# ==========================================
# CELL 3: ZENODO DOWNLOAD (Rule 3 — stable URL, Rule 2 — research data repository)
# Run once, or when a new model version is published to Zenodo.
# ==========================================

# Read authentication token from environment variable (NEVER hardcode tokens here)
ACCESS_TOKEN = os.environ.get("ZENODO_TOKEN", None)

if ACCESS_TOKEN:
    print("Zenodo access token loaded from environment variable ZENODO_TOKEN.")
else:
    print("ZENODO_TOKEN not set. Attempting unauthenticated download (open-access records only).")

print(f"\n--- Downloading from Zenodo record {ZENODO_RECORD_ID} ---")

for filename in ZENODO_FILES:
    local_zip = ZENODO_DIR / filename

    # Skip if already downloaded (idempotent — safe to re-run)
    if local_zip.exists():
        print(f"Already exists, skipping download: {filename}")
        continue

    # Full query URL (showing complete API request per Rule 4 / authentication guidelines)
    url    = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}/files/{filename}/content"
    params = {"access_token": ACCESS_TOKEN} if ACCESS_TOKEN else {}

    print(f"Downloading: {filename}")
    print(f"  URL: {url}")

    response = requests.get(url, params=params, stream=True, timeout=120)

    if response.status_code == 200:
        with open(local_zip, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"  Saved: {local_zip}")
    else:
        print(f"  Failed (HTTP {response.status_code}). Check ZENODO_TOKEN and record ID.")

print("\n--- Extracting zip files ---")

for filename in ZENODO_FILES:
    local_zip = ZENODO_DIR / filename
    if local_zip.exists():
        print(f"Extracting: {filename}")
        with zipfile.ZipFile(local_zip, "r") as zip_ref:
            zip_ref.extractall(ZENODO_DIR)
        print(f"  Extracted to: {ZENODO_DIR}")
    else:
        print(f"  Skipping (not found): {filename}")

# Verify expected folders exist after extraction
print("\n--- Verifying extracted content ---")
for path, label in [(TXTINOUT_DIR, "TxtInOut"), (SHAPE_DIR, "Shapes")]:
    status = "FOUND" if Path(path).exists() else "NOT FOUND — check zip folder name"
    print(f"  {label}: {path}  [{status}]")

---
# Part B — Dynamic meteorological forcing

**Choose one option by setting `FORCING_OPTION` in Cell 2.**  
All three options produce the same output variables (pcp, tmax, tmin, hmd, slr) so SE2 works identically regardless of which you chose.

| Option | Best suited for | Data lag |
|--------|----------------|----------|
| FMI_POINT (default) | Near-real-time operational prediction; single gauge | ~1 day |
| FMI_GRID | Spatially distributed analysis across the catchment | ~1–3 days |
| ERA5 | Long historical periods; catchments without FMI stations | ~5 days to months |

---
## Option 1: FMI Point Station + Harmonie NWP (Rule 4 — OGC WFS API)

### Cell 4a — FMI observations download (historical, incremental)

**Full API query:** `fmi::observations::weather::daily::multipointcoverage`, fmisid=101887  
**Radiation API:** `fmi::observations::radiation::multipointcoverage`, fmisid=101932  
Smart state detection reads the last date in the existing `.pcp` file and fetches only missing days.

In [ ]:
# ==========================================
# CELL 4a: FMI POINT — OBSERVATIONS (Option 1)
# Rule 4: full API query shown; no authentication required
# ==========================================

df_obs = pd.DataFrame()    # initialise as empty; filled below if option selected

if FORCING_OPTION == "FMI_POINT":

    # --- 1. Smart state detection: find the last date already in TxtInOut ---
    obs_archives      = sorted(glob.glob(os.path.join(BASE_DIR, "TxtInOut_Obs_Only_*")))
    source_obs_folder = obs_archives[-1] if obs_archives else str(TXTINOUT_DIR)

    def get_last_date(folder):
        """Read the last Julian date line from the first .pcp file in a SWAT+ folder."""
        pcp_files = glob.glob(os.path.join(folder, "*.pcp"))
        if not pcp_files:
            return None
        with open(pcp_files[0], 'r') as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
            if len(lines) < 4:
                return None
            parts = lines[-1].split()
            return datetime.datetime(int(parts[0]), 1, 1) + datetime.timedelta(int(parts[1]) - 1)

    last_obs_date = get_last_date(source_obs_folder)
    fetch_start   = last_obs_date + datetime.timedelta(days=1)
    fetch_end     = datetime.datetime.now() - datetime.timedelta(days=1)   # yesterday

    print(f"Source folder : {os.path.basename(source_obs_folder)}")
    print(f"Last obs date : {last_obs_date.date()}")
    print(f"Fetch range   : {fetch_start.date()} to {fetch_end.date()}")

    obs_records = []

    if fetch_start <= fetch_end:
        t0 = time.time()

        # Build 7-day chunks (FMI WFS API limit per single request)
        curr, chunks = fetch_start, []
        while curr <= fetch_end:
            chunks.append((curr, min(curr + datetime.timedelta(days=6), fetch_end)))
            curr = chunks[-1][1] + datetime.timedelta(days=1)

        for i, (s, e) in enumerate(chunks):
            s_str = s.strftime("%Y-%m-%dT00:00:00Z")
            e_str = e.strftime("%Y-%m-%dT23:59:59Z")
            try:
                # Full API queries (shown in full per Rule 4 / authentication guidelines)
                o_daily  = download_stored_query(
                    "fmi::observations::weather::daily::multipointcoverage",
                    args=[f"fmisid={FMI_STATION_ID_OBS}",
                          "starttime=" + s_str, "endtime=" + e_str]
                )
                o_hourly = download_stored_query(
                    "fmi::observations::weather::multipointcoverage",
                    args=[f"fmisid={FMI_STATION_ID_OBS}",
                          "starttime=" + s_str, "endtime=" + e_str]
                )
                o_rad    = download_stored_query(
                    "fmi::observations::radiation::multipointcoverage",
                    args=[f"fmisid={FMI_STATION_ID_RAD}",
                          "starttime=" + s_str, "endtime=" + e_str]
                )

                for t in o_daily.data.keys():
                    dt     = pd.to_datetime(t).tz_localize(None)
                    st_key = list(o_daily.data[t].keys())[0]
                    row    = o_daily.data[t][st_key]

                    pcp_val  = row.get('Precipitation amount', {}).get('value')
                    tmax_val = row.get('Maximum temperature',  {}).get('value')
                    tmin_val = row.get('Minimum temperature',  {}).get('value')

                    # Aggregate hourly humidity → daily mean fraction [0-1]
                    h_v = [
                        float(dh[list(dh.keys())[0]].get('Relative humidity', {}).get('value'))
                        for th, dh in o_hourly.data.items()
                        if dt <= pd.to_datetime(th).tz_localize(None) <= (dt + datetime.timedelta(hours=23))
                    ]
                    # Aggregate hourly radiation → daily sum in MJ/m²/day
                    r_v = [
                        float(ds[list(ds.keys())[0]].get('Global radiation', {}).get('value'))
                        for ts, ds in o_rad.data.items()
                        if dt <= pd.to_datetime(ts).tz_localize(None) <= (dt + datetime.timedelta(hours=23))
                    ]

                    obs_records.append({
                        'time': dt,
                        'pcp' : pcp_val,
                        'tmax': tmax_val,
                        'tmin': tmin_val,
                        'hmd' : (np.mean(h_v) / 100.0) if h_v else np.nan,
                        'slr' : (np.sum(r_v) * 0.00006) if r_v else np.nan
                    })

            except Exception as err:
                print(f"\nWarning: chunk {s.date()} to {e.date()} failed: {err}")

            print(f"Progress: {((i+1)/len(chunks)*100):.1f}%  |  Elapsed: {int(time.time()-t0)}s", end="\r")

        df_obs = pd.DataFrame(obs_records).set_index('time').sort_index() if obs_records else pd.DataFrame()
        print(f"\nFMI point observations downloaded: {len(df_obs)} new days.")

    else:
        print("FMI point data already up to date. df_obs is empty.")

else:
    print(f"Skipping Option 1 (FORCING_OPTION = '{FORCING_OPTION}').")

### Cell 4b — FMI Harmonie NWP forecast download (Option 1 only)

**Full API query:** `fmi::forecast::harmonie::surface::point::simple`, latlon=66.364,29.316  
No authentication required. Returns ~60 hours of hourly forecast data.

In [ ]:
# ==========================================
# CELL 4b: FMI POINT — HARMONIE NWP FORECAST (Option 1)
# Rule 4: full API query shown
# ==========================================

df_forecast = pd.DataFrame()   # initialise as empty

if FORCING_OPTION == "FMI_POINT":

    print("Fetching Harmonie NWP forecast from FMI Open Data API...")

    # Full WFS request parameters (shown in full per Rule 4)
    forecast_params = {
        "service"        : "WFS",
        "version"        : "2.0.0",
        "request"        : "getFeature",
        "storedquery_id" : "fmi::forecast::harmonie::surface::point::simple",
        "latlon"         : FORECAST_LATLON,
        "parameters"     : "Temperature,Humidity,Precipitation1h,RadiationGlobal"
    }

    forecast_response = requests.get("http://opendata.fmi.fi/wfs", params=forecast_params)
    root = ET.fromstring(forecast_response.content)

    f_list = []
    ns = {'BsWfs': 'http://xml.fmi.fi/schema/wfs/2.0'}
    for el in root.findall('.//BsWfs:BsWfsElement', ns):
        f_list.append({
            'Time' : pd.to_datetime(el.find('{http://xml.fmi.fi/schema/wfs/2.0}Time').text),
            'Param': el.find('{http://xml.fmi.fi/schema/wfs/2.0}ParameterName').text,
            'Val'  : float(el.find('{http://xml.fmi.fi/schema/wfs/2.0}ParameterValue').text)
        })

    f_df = pd.DataFrame(f_list).pivot_table(index='Time', columns='Param', values='Val')

    # Aggregate hourly forecast to daily SWAT+ format
    df_forecast = pd.DataFrame()
    df_forecast['tmax'] = f_df['Temperature'].resample('1D').max()
    df_forecast['tmin'] = f_df['Temperature'].resample('1D').min()
    df_forecast['pcp']  = f_df['Precipitation1h'].resample('1D').sum()          # mm/day
    df_forecast['hmd']  = f_df['Humidity'].resample('1D').mean() / 100.0         # fraction
    df_forecast['slr']  = f_df['RadiationGlobal'].resample('1D').mean() * 0.0864 # MJ/m2/day
    df_forecast.index   = df_forecast.index.tz_localize(None)

    # Keep only future dates (today onwards)
    df_forecast = df_forecast[df_forecast.index >= pd.Timestamp.now().normalize()]

    print(f"Harmonie forecast downloaded: {len(df_forecast)} days.")
    print(df_forecast)

else:
    print(f"Skipping Harmonie forecast (FORCING_OPTION = '{FORCING_OPTION}').")

---
## Option 2: FMI Gridded 1 km Analysis (Rule 3 — stable S3 URL)

The FMI publishes daily gridded meteorological analysis at 1 km resolution covering all of Finland as NetCDF files on a public Amazon S3 bucket. No authentication is required. Files are updated daily with a lag of 1–3 days.

**Stable URL pattern:**  
`http://fmi-gridded-obs-daily-1km.s3-website-eu-west-1.amazonaws.com/Netcdf/{Measurand}/{measurand}_{year}.nc`

**Variables downloaded:**

| FMI variable | SWAT+ use | Unit | Conversion |
|---|---|---|---|
| `RRday` | Precipitation (pcp) | mm/day | × 1.0 |
| `Tmax` | Max temperature | °C | × 1.0 |
| `Tmin` | Min temperature | °C | × 1.0 |
| `Rh` | Relative humidity (hmd) | % | ÷ 100 → fraction |
| `Globrad` | Solar radiation (slr) | W/m² | × 0.001 → kW/m²; daily sum in MJ/m² |

Station coordinates are read from `weather-sta.cli` and projected from WGS84 to ETRS-TM35FIN (EPSG:3067) to match the NetCDF grid.

In [ ]:
# ==========================================
# CELL 5a: FMI GRID — DOWNLOAD NetCDF FILES (Option 2)
# Rule 3: stable S3 URL; no authentication required
# ==========================================

if FORCING_OPTION == "FMI_GRID":

    current_year = datetime.datetime.now().year
    years = list(range(FMI_GRID_START_YR, current_year + 1))

    print(f"Downloading FMI Gridded 1km data: {FMI_GRID_VARS} for years {years}")
    print(f"Base URL: {FMI_GRID_S3_URL}")

    for measurand in FMI_GRID_VARS:
        var_dir = FMI_GRID_DIR / measurand
        var_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n--- {measurand} ---")
        for year in years:
            filename  = f"{measurand.lower()}_{year}.nc"
            url       = f"{FMI_GRID_S3_URL}{measurand}/{filename}"   # full URL shown (Rule 4)
            save_path = var_dir / filename

            if save_path.exists():
                print(f"  Already downloaded, skipping: {filename}")
                continue

            print(f"  Downloading: {filename}")
            print(f"  URL: {url}")
            try:
                response = requests.get(url, timeout=60, stream=True)
                if response.status_code == 200:
                    with open(save_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                    print(f"  Saved: {save_path}")
                elif response.status_code == 404:
                    print(f"  Not found on server (file not yet generated for this year): {filename}")
                else:
                    print(f"  Download failed (HTTP {response.status_code}): {filename}")
            except Exception as e:
                print(f"  Error downloading {filename}: {e}")

    print("\nFMI grid download complete.")

else:
    print(f"Skipping Option 2 download (FORCING_OPTION = '{FORCING_OPTION}').")

### Cell 5b — Extract FMI gridded data at SWAT+ station locations and build df_obs

Station coordinates are read from `weather-sta.cli`. For each station, the nearest NetCDF grid cell is selected using `xarray.sel(..., method='nearest')`. NaN values (gaps in the gridded product) are filled with the mean of neighbouring station values.

In [ ]:
# ==========================================
# CELL 5b: FMI GRID — EXTRACT AT STATION LOCATIONS (Option 2)
# ==========================================

if FORCING_OPTION == "FMI_GRID":

    # Coordinate transformer: WGS84 (lat/lon) → ETRS-TM35FIN (northing/easting)
    # The FMI 1 km grid uses ETRS-TM35FIN (EPSG:3067)
    transformer = Transformer.from_crs("epsg:4326", "epsg:3067", always_xy=True)

    # Read station list from weather-sta.cli
    cli_file = TXTINOUT_DIR / "weather-sta.cli"
    stations = []
    with open(cli_file, 'r') as f:
        for line in f.readlines()[2:]:
            parts = line.split()
            if len(parts) >= 6:
                stations.append({
                    'name': parts[0],
                    'pcp' : parts[2], 'tmp': parts[3],
                    'slr' : parts[4], 'hmd': parts[5]
                })

    current_year = datetime.datetime.now().year
    years        = list(range(FMI_GRID_START_YR, current_year + 1))

    # Variable mapping: (SWAT+ key, FMI measurand folder, FMI variable name, scale factor)
    var_map = [
        ('pcp',  'RRday',  'RRday',  1.0),    # precipitation mm/day
        ('hmd',  'Rh',     'Rh',     0.01),   # relative humidity → fraction
        ('slr',  'Globrad','Globrad',0.001),   # W/m² → kW/m² (daily sum gives MJ/m²)
    ]

    def fill_nan_with_neighbours(series, neighbour_series_list):
        """Replace NaN values with mean of valid neighbouring station values on the same date."""
        if not series.isna().any():
            return series
        for dt in series[series.isna()].index:
            valid = [s[dt] for s in neighbour_series_list if dt in s.index and not pd.isna(s[dt])]
            series[dt] = np.mean(valid) if valid else 0.0
        return series

    # --- Phase 1: Load all grid data for all stations ---
    print("Phase 1: Loading FMI grid data for all stations...")
    all_data = {}

    for st in stations:
        # Parse station coordinates from name (format: st6645_2901 → lat=66.45, lon=29.01)
        nums = re.findall(r"(\d+)", st['name'])
        lat  = float(nums[0][:2] + "." + nums[0][2:])
        lon  = float(nums[1][:2] + "." + nums[1][2:])
        e, n = transformer.transform(lon, lat)   # lon, lat → easting, northing

        all_data[st['name']] = {}

        for key, meas, var, scale in var_map:
            data_list = []
            for yr in years:
                nc_path = FMI_GRID_DIR / meas / f"{meas.lower()}_{yr}.nc"
                if nc_path.exists():
                    ds    = xr.open_dataset(nc_path)
                    y_dim = next(d for d in ['Lat', 'lat', 'n'] if d in ds.dims)
                    x_dim = next(d for d in ['Lon', 'lon', 'e'] if d in ds.dims)
                    series = ds[var].sel({y_dim: n, x_dim: e}, method='nearest').to_series()
                    data_list.append(series * scale)
                    ds.close()
            if data_list:
                all_data[st['name']][key] = pd.concat(data_list)

        # Temperature (Tmax and Tmin loaded separately)
        for temp_key, meas in [('tmax', 'Tmax'), ('tmin', 'Tmin')]:
            data_list = []
            for yr in years:
                nc_path = FMI_GRID_DIR / meas / f"{meas.lower()}_{yr}.nc"
                if nc_path.exists():
                    ds    = xr.open_dataset(nc_path)
                    y_dim = next(d for d in ['Lat', 'lat', 'n'] if d in ds.dims)
                    x_dim = next(d for d in ['Lon', 'lon', 'e'] if d in ds.dims)
                    series = ds[meas].sel({y_dim: n, x_dim: e}, method='nearest').to_series()
                    data_list.append(series)
                    ds.close()
            if data_list:
                all_data[st['name']][temp_key] = pd.concat(data_list)

    # --- Phase 2: Gap-fill NaN and produce df_obs for the primary (first) station ---
    print("Phase 2: Gap-filling and building df_obs...")
    primary_station = stations[0]['name']
    neighbour_names = [s['name'] for s in stations[1:]]

    records = {}
    for key in ['pcp', 'tmax', 'tmin', 'hmd', 'slr']:
        if key in all_data[primary_station]:
            s = all_data[primary_station][key].copy()
            neighbours = [all_data[n][key] for n in neighbour_names if key in all_data.get(n, {})]
            records[key] = fill_nan_with_neighbours(s, neighbours)

    df_obs = pd.DataFrame(records)
    df_obs.index = pd.to_datetime(df_obs.index).tz_localize(None)
    df_obs = df_obs.sort_index()

    # Precipitation: clamp negatives to 0 (FMI trace = -1.0)
    df_obs['pcp'] = df_obs['pcp'].clip(lower=0.0)

    # Note: FMI grid has no NWP forecast — df_forecast remains empty for this option
    df_forecast = pd.DataFrame()

    print(f"FMI grid df_obs built: {len(df_obs)} days from {df_obs.index.min().date()} to {df_obs.index.max().date()}")
    print(df_obs.tail(5))

else:
    print(f"Skipping Option 2 extraction (FORCING_OPTION = '{FORCING_OPTION}').")

---
## Option 3: ERA5 Reanalysis via Copernicus CDS API (Rule 4 — full API query)

ERA5 is the ECMWF fifth-generation global climate reanalysis, available via the Copernicus Climate Data Store (CDS).  
It is best suited for long historical periods or catchments where FMI observations are sparse.

**Authentication:** Register at https://cds.climate.copernicus.eu and configure `~/.cdsapirc` per https://cds.climate.copernicus.eu/how-to-api  
The `cdsapi` library reads credentials from that file — no tokens appear in this notebook.

**Important:** ERA5 has an approximate lag of 5 days for the preliminary back extension and several months for the final reanalysis. It is not suitable for the Harmonie-based near-real-time forecast component.

In [ ]:
# ==========================================
# CELL 6: ERA5 DOWNLOAD VIA CDS API (Option 3)
# Rule 4: full API query shown below
# Authentication: credentials in ~/.cdsapirc (see README)
# ==========================================

if FORCING_OPTION == "ERA5":

    import cdsapi   # pip install cdsapi

    # CDS dataset identifier
    ERA5_DATASET = "reanalysis-era5-single-levels"

    # Full API request (shown in its entirety per Rule 4 / authentication guidelines)
    ERA5_REQUEST = {
        "product_type"  : ["reanalysis"],
        "variable"      : [
            "2m_temperature",
            "total_precipitation",
            "surface_solar_radiation_downwards",
            "2m_dewpoint_temperature"        # used to derive relative humidity
        ],
        "year"          : ERA5_YEARS,
        "month"         : [f"{m:02d}" for m in range(1, 13)],
        "day"           : [f"{d:02d}" for d in range(1, 32)],
        "time"          : [f"{h:02d}:00" for h in range(0, 24)],   # hourly
        "data_format"   : "netcdf",
        "download_format": "unarchived",
        "area"          : ERA5_AREA   # [North, West, South, East] bounding box
    }

    era5_output = ERA5_DIR / "era5_oulanka_catchment.nc"

    if era5_output.exists():
        print(f"ERA5 file already exists, skipping download: {era5_output}")
    else:
        print(f"Submitting ERA5 request to Copernicus CDS API...")
        print(f"Dataset: {ERA5_DATASET}")
        print(f"Area   : {ERA5_AREA}")
        print(f"Years  : {ERA5_YEARS[0]} to {ERA5_YEARS[-1]}")
        print(f"Output : {era5_output}")

        client = cdsapi.Client()   # reads credentials from ~/.cdsapirc automatically
        client.retrieve(ERA5_DATASET, ERA5_REQUEST).download(target=str(era5_output))
        print(f"ERA5 download complete: {era5_output}")

    # --- Process ERA5 NetCDF to daily SWAT+ format at the Oulanka outlet point ---
    print("\nProcessing ERA5 data to daily SWAT+ format at outlet point (66.364°N, 29.316°E)...")
    ds_era5 = xr.open_dataset(era5_output)

    # Select the nearest grid cell to the Oulanka outlet
    point = ds_era5.sel(latitude=66.364, longitude=29.316, method='nearest')

    daily = point.resample(valid_time='1D')

    df_obs = pd.DataFrame(index=daily.mean()['valid_time'].values)

    # Convert units to SWAT+ expected format
    df_obs['tmax'] = daily.max()['t2m'].values - 273.15          # K → °C
    df_obs['tmin'] = daily.min()['t2m'].values - 273.15          # K → °C
    df_obs['pcp']  = daily.sum()['tp'].values * 1000.0           # m → mm
    df_obs['slr']  = daily.sum()['ssrd'].values / 1e6            # J/m² → MJ/m²

    # Relative humidity from dewpoint: Magnus formula
    t2m  = daily.mean()['t2m'].values - 273.15
    d2m  = daily.mean()['d2m'].values - 273.15
    es_t = 6.112 * np.exp(17.67 * t2m  / (t2m  + 243.5))
    es_d = 6.112 * np.exp(17.67 * d2m  / (d2m  + 243.5))
    df_obs['hmd'] = np.clip(es_d / es_t, 0.0, 1.0)              # fraction [0-1]

    df_obs.index = pd.to_datetime(df_obs.index).tz_localize(None)
    df_obs['pcp'] = df_obs['pcp'].clip(lower=0.0)               # clamp negatives
    ds_era5.close()

    # ERA5 has no NWP forecast extension
    df_forecast = pd.DataFrame()

    print(f"ERA5 df_obs built: {len(df_obs)} days from {df_obs.index.min().date()} to {df_obs.index.max().date()}")
    print(df_obs.tail(5))

else:
    print(f"Skipping Option 3 (FORCING_OPTION = '{FORCING_OPTION}').")

---
## Summary of outputs

| Variable | Type | Contents | Produced by | Used in |
|----------|------|----------|-------------|--------|
| `TXTINOUT_DIR` | `Path` | Extracted SWAT+ model input folder | Part A (Zenodo) | SE2, SE3 |
| `SHAPE_DIR` | `Path` | GIS shapefiles (watershed, subbasins, rivers) | Part A (Zenodo) | SE4 |
| `df_obs` | `pd.DataFrame` | Historical daily forcing (new days only, may be empty) | Part B (any option) | SE2 |
| `df_forecast` | `pd.DataFrame` | 2–3 day Harmonie NWP forecast (Option 1 only; empty for 2 and 3) | Option 1 only | SE2 |

All DataFrames share the same column schema:  
`pcp` (mm/day) · `tmax` (°C) · `tmin` (°C) · `hmd` (fraction 0–1) · `slr` (MJ/m²/day)

SE2 reads `df_obs` and `df_forecast` identically regardless of which forcing option produced them.

---
## Need help?
Search, ask, and answer data access questions at https://github.com/orgs/DigitalWaters-fi/discussions  
Tag **#dataaccess**